**Bowler_model:** overs_bowled, runs_conceded, wickets_taken, runs_conceded_in_pp, runs_conceded_in_mo, runs_conceded_in_death,runs_conceded_in_wide,runs_conceded_in_noball

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df_matches = spark.read.format('csv')\
            .option('header', 'true')\
            .option('inferSchema', 'true')\
            .load('/Workspace/Users/asyedshabeeradhnan01@gmail.com/Data-science/raw/ipl_matches.csv')

In [0]:
df_eachball_info = spark.read.format('csv')\
               .option('header', 'true')\
               .option('inferSchema', 'true')\
               .load('/Workspace/Users/asyedshabeeradhnan01@gmail.com/Data-science/raw/ipl_ball_by_ball.csv')

In [0]:
df_period = df_eachball_info.withColumn('season',when(col('season').rlike(r"^\d{4}/\d{2}$"),year(col('date'))).otherwise(col('season')))\
              .withColumn('season', col('season').cast(IntegerType()))\
              .filter(col('season') >= 2020)

In [0]:
df_sixseason_players = df_period.select('season','bowler').distinct()\
                            .groupBy('bowler').agg(count('season').alias('total_season_played'))\
                            .filter(col('total_season_played') >= 6)

In [0]:
df_join = df_period.join(df_sixseason_players, how='left_semi', on=['bowler'])

In [0]:
df_no_of_balls_bowled = df_join.groupBy('over','bowler')\
                    .agg(count('over').alias('balls_bowled')\
                    ,sum('total_runs').alias('runs_given')\
                    ,sum('is_wicket').alias('wickets')\
                    ,sum(when(col('is_powerplay') == 1,col('total_runs'))).alias('runs_in_po')\
                    ,sum(when(col('is_middle_overs') == 1,col('total_runs'))).alias('runs_in_mo')\
                    ,sum(when(col('is_death_overs') == 1,col('total_runs'))).alias('runs_in_do')\
                    ,sum('wides').alias('wides')\
                    ,sum('noballs').alias('noballs'))

In [0]:
df_no_of_over_bowled = df_no_of_balls_bowled.groupBy('bowler')\
                    .agg(count('bowler').alias('over_bowled')\
                    ,sum('runs_given').alias('runs_given')\
                    ,sum('wickets').alias('wickets')\
                    ,sum('runs_in_po').alias('runs_in_po')\
                    ,sum('runs_in_mo').alias('runs_in_mo')\
                    ,sum('runs_in_do').alias('runs_in_do')\
                    ,sum('wides').alias('wides')\
                    ,sum('noballs').alias('noballs'))\
                    .orderBy(col('over_bowled').desc(),col('runs_given').asc())

In [0]:
df_no_of_over_bowled.display()